# Task 4

> *Run the above various algorithms in 3) on MSH-WSD dataset and report the average accuracy on all acronyms
> and average accuracy on all regular terms.*

This task combines Task 2 and Task 3: we use the MSH-WSD data and the manual CUI → WordNet mapping of Task 2, and replace NLTK Lesk with the pywsd algorithms of Task 3. The functions shared with the other tasks (reading the data, the mapping, the translation of the pywsd answers, the context window and the accuracy) are in `wsd_utils.py`.

In [7]:
import sys
sys.path.append("..")                                  # wsd_utils.py is in the parent folder
from wsd_utils import (ALL_ACRONYMS, ALL_TERMS, MAPPING, load_instances, wordnet_senses,
                       to_nltk, pywsd_form, context_window, sample_instances, accuracy, mean)

import json
import time
from pathlib import Path
import pandas as pd

import nltk
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
from nltk.wsd import lesk

from pywsd.lesk import original_lesk, simple_lesk, adapted_lesk, cosine_lesk
from pywsd.similarity import max_similarity
from pywsd.baseline import max_lemma_count

**The words**

We evaluate the algorithms on the words of Task 2 (both selections together, 39 words), because they are the only ones with a **manual CUI → WordNet mapping**, i.e. the correct answers translated into WordNet senses. We split them into acronyms and regular terms with the same rule as Task 2, because the task asks for a separate accuracy for each group.

In [8]:
ACRONYMS = [w for w in ALL_ACRONYMS if w in MAPPING]
TERMS    = [w for w in ALL_TERMS if w in MAPPING]

print(len(ACRONYMS), "acronyms:", ACRONYMS)
print(len(TERMS), "terms:", TERMS)

20 acronyms: ['ADP', 'ALS', 'ANA', 'CAD', 'CAM', 'CDA', 'CI', 'CLS', 'Ca', 'DDD', 'DON', 'FAS', 'GAG', 'MRS', 'PVC', 'RA', 'SARS', 'Staph', 'TNT', 'US']
19 terms: ['B-Cell Leukemia', 'Borrelia', 'Callus', 'Cell', 'Cement', 'Cilia', 'ERUPTION', 'Exercises', 'Follicles', 'Ganglion', 'Medullary', 'Milk', 'Nurse', 'Phosphorus', 'Pneumocystis', 'Respiration', 'Sodium', 'Synapsis', 'Ventricles']


Another mapping is needed, between the WordNet version used by NLTK and the one used by pywsd. 
Our `MAPPING` gives the correct answers with the NLTK names, while the pywsd algorithms answer with their own IDs, so we need to translate them. To do that we use the **sense key**, a code that identifies the same sense in both versions.

Before looking a word up, pywsd lemmatizes it and distinguishes upper and lower case, so the same word can give different senses. The function "pywsd_form" tries a few spellings and keeps the one that finds the most senses of our `MAPPING`.

In [9]:
# Spelling chosen for each word
for word in ACRONYMS + TERMS:
    print(f"{word:16} -> {pywsd_form(word)!r}")

ADP              -> 'ADP'
ALS              -> 'ALS'
ANA              -> 'ANA'
CAD              -> 'cad'
CAM              -> 'CAM'
CDA              -> 'CDA'
CI               -> 'CI'
CLS              -> 'CLS'
Ca               -> 'ca'
DDD              -> 'DDD'
DON              -> 'DON'
FAS              -> 'FAS'
GAG              -> 'GAG'
MRS              -> 'MRS'
PVC              -> 'PVC'
RA               -> 'ra'
SARS             -> 'SARS'
Staph            -> 'Staph'
TNT              -> 'TNT'
US               -> 'US'
B-Cell Leukemia  -> 'B-Cell Leukemia'
Borrelia         -> 'Borrelia'
Callus           -> 'Callus'
Cell             -> 'Cell'
Cement           -> 'Cement'
Cilia            -> 'Cilia'
ERUPTION         -> 'ERUPTION'
Exercises        -> 'exercises'
Follicles        -> 'follicles'
Ganglion         -> 'Ganglion'
Medullary        -> 'Medullary'
Milk             -> 'milk'
Nurse            -> 'Nurse'
Phosphorus       -> 'phosphorus'
Pneumocystis     -> 'Pneumocystis'
Respiration      -

**Context window and sample**

The similarity-based methods compare every context word with every sense of the target word, so on a whole abstract they are very slow. We therefore use a **context window**: only the 10 words before and the 10 words after the target word, found thanks to the `<e>...</e>` tags. The closest words are usually also the most useful clues, and the same window is used for all the methods, so the comparison is fair.

For the same reason we do not use all the abstracts, but a random sample of **10 abstracts per sense**. Sampling the same number per sense keeps the senses balanced: with a random sample, a word with 150 abstracts of one sense and 50 of the other could end up with almost only the first sense, and a method that always chooses it would look better than it is. The sample is fixed (`seed=0`), so it is the same every time the notebook is run.

In [10]:
example = load_instances("AA")[0]
print("Full abstract :", len(example["context"].split()), "words")
print("Window        :", context_window(example["tagged"]))

n_abstracts = sum(len(sample_instances(w)) for w in ACRONYMS + TERMS)
print("\nAbstracts in the sample:", n_abstracts)

Full abstract : 262 words
Window        : effect of the duration of prefeeding on prececal amino acid AA digestibility of wheat distillers dried grains with solubles DDGS in

Abstracts in the sample: 810


Now we put all the methods in a single dictionary `METHODS`, so that we can run them with one loop, and we
write a function `predict` that runs one method on one context window and returns its answer as an NLTK
synset name.

The methods are the ones of Task 3 (Lesk family, similarity-based methods and Max lemma count), plus:
- **NLTK Lesk**, the algorithm of Task 2, as a reference. It uses the NLTK WordNet directly, so its answer
  does not need to be translated;
- **Resnik (fixed)**: in Task 3 we found that pywsd sorts the Resnik ranking in ascending order and returns
  the **least** similar sense. `resnik_fixed` takes the full ranking and returns the sense with the
  **highest** score. We keep both versions to see how much the bug changes the results.

In [11]:
def resnik_fixed(context, word):
    """Resnik with the ranking sorted correctly (highest score = most similar)."""
    ranking = max_similarity(context, word, option="res", best=False)
    if not ranking:
        return None
    return max(ranking, key=lambda x: x[0])[1]


METHODS = {
    "Original Lesk":    lambda ctx, w: original_lesk(ctx, w),
    "Simple Lesk":      lambda ctx, w: simple_lesk(ctx, w),
    "Adapted Lesk":     lambda ctx, w: adapted_lesk(ctx, w),
    "Cosine Lesk":      lambda ctx, w: cosine_lesk(ctx, w),
    "Path":             lambda ctx, w: max_similarity(ctx, w, option="path"),
    "Wu-Palmer":        lambda ctx, w: max_similarity(ctx, w, option="wup"),
    "Leacock-Chodorow": lambda ctx, w: max_similarity(ctx, w, option="lch"),
    "Resnik (pywsd)":   lambda ctx, w: max_similarity(ctx, w, option="res"),
    "Resnik (fixed)":   resnik_fixed,
    "Jiang-Conrath":    lambda ctx, w: max_similarity(ctx, w, option="jcn"),
    "Lin":              lambda ctx, w: max_similarity(ctx, w, option="lin"),
    "Max lemma count":  lambda ctx, w: max_lemma_count(w),
}
ALL_METHODS = ["NLTK Lesk"] + list(METHODS)


def predict(method, context, word):                                   
    """Run one method on one context and return its answer as an NLTK synset name (or None)."""
    try:
        if method == "NLTK Lesk":
            senses = wordnet_senses(word)
            if not senses:
                return None
            answer = lesk(context.lower().split(), word.lower(), synsets=senses)
            return answer.name() if answer else None
        answer = METHODS[method](context, pywsd_form(word))           
        return to_nltk(answer)
    except Exception:
        return None

Now we run all the methods on every abstract of the sample. For each abstract we store its correct sense (CUI)
and the answer of every method; the accuracy is computed later from these stored answers, without running the
algorithms again.

In [12]:
CACHE = Path("task4_predictions.json")   

predictions = json.loads(CACHE.read_text()) if CACHE.exists() else {}

for word in ACRONYMS + TERMS:
    # Skip words that have already been processed and saved in the cache
    if word in predictions:        
        continue

    start = time.time()
    rows = []
    for inst in sample_instances(word):
        ctx = context_window(inst["tagged"])
        answers = {method: predict(method, ctx, word) for method in ALL_METHODS}
        rows.append({"cui": inst["cui"], "answers": answers})

    predictions[word] = rows
    CACHE.write_text(json.dumps(predictions))  # save after every word
    print(f"{word:16} {len(rows):3} abstracts  {time.time() - start:6.1f} s")

print("Done.")

Done.


## Results
Now we compute the accuracy of each method, separately for acronyms and regular terms. We use two measures:
- **accuracy**: correct answers / all abstracts of the word. If the correct sense does not exist in WordNet (empty list in `MAPPING`), no method can find it, so the answer is always wrong.
- **accuracy on covered senses**: the same, but only on the abstracts whose correct sense exists in WordNet. It shows how good the method is when a correct answer is possible.

For each method we first compute the accuracy of every word, and then the average over the words of the group. Words with no covered sense at all (e.g. DDD) are not counted in the second measure.

In [13]:
table = []
for method in ALL_METHODS:
    line = {"method": method}
    for group_name, group in [("acronyms", ACRONYMS), ("terms", TERMS)]:
        line[f"{group_name} acc"] = mean(accuracy(predictions[w], w, method) for w in group)
        line[f"{group_name} cov"] = mean(accuracy(predictions[w], w, method, covered_only=True) for w in group)
    table.append(line)

results = pd.DataFrame(table).set_index("method")
results.round(3)

,acronyms acc,acronyms cov,terms acc,terms cov
method,,,,
NLTK Lesk,0.225,0.750,0.347,0.565
Original Lesk,0.211,0.705,0.334,0.529
Simple Lesk,0.242,0.815,0.353,0.550
Adapted Lesk,0.242,0.815,0.353,0.550
Cosine Lesk,0.230,0.790,0.368,0.568
Path,0.257,0.925,0.363,0.582
Wu-Palmer,0.250,0.905,0.345,0.562
Leacock-Chodorow,0.249,0.900,0.358,0.576
Resnik (pywsd),0.188,0.680,0.253,0.447


**Observations:**
- The accuracy is low for all methods (0.19–0.37). The main reason is that many medical senses do not exist in WordNet, so no method can find them.
- On covered senses the accuracy is much higher, especially for acronyms. But this is misleading: only a few acronyms are covered, and most of them have only one sense in WordNet, so pywsd has only one choice.
- On terms, the similarity methods **Lin**, **Path** and **Leacock-Chodorow** are slightly better than the Lesk methods, but the differences are small and we use only 20 abstracts per word.
- **Resnik (pywsd)** is the worst method; after our fix, **Resnik (fixed)** becomes one of the best on acronyms. This confirms the bug found in Task 3.
- **Simple Lesk** and **Adapted Lesk** give exactly the same results.
- **Max lemma count** is the worst on terms: Open English WordNet has no lemma counts, so it always returns the first sense.

In [14]:
SHOW = ["NLTK Lesk", "Simple Lesk", "Cosine Lesk", "Wu-Palmer", "Lin", "Resnik (fixed)", "Max lemma count"]

table = []
for group_name, group in [("acronym", ACRONYMS), ("term", TERMS)]:
    for w in group:
        rows = predictions[w]
        line = {"word": w, "group": group_name,
                "covered": sum(bool(MAPPING[w][r["cui"]]) for r in rows) / len(rows)}
        for m in SHOW:
            line[m] = accuracy(rows, w, m)
        table.append(line)

per_word = pd.DataFrame(table).set_index(["group", "word"])
per_word.round(2)

covered  NLTK Lesk  Simple Lesk  Cosine Lesk  \
group   word                                                            
acronym ADP                 1.00       0.55         0.55         0.55   
        ALS                 0.50       0.50         0.50         0.50   
        ANA                 0.00       0.00         0.00         0.00   
        CAD                 0.00       0.00         0.00         0.00   
        CAM                 0.00       0.00         0.00         0.00   
        CDA                 0.00       0.00         0.00         0.00   
        CI                  0.00       0.00         0.00         0.00   
        CLS                 0.00       0.00         0.00         0.00   
        Ca                  0.50       0.25         0.30         0.30   
        DDD                 0.00       0.00         0.00         0.00   
        DON                 0.00       0.00         0.00         0.00   
        FAS                 0.50       0.45         0.50         0.50   
        GAG                 0.00       0.00         0.00         0.00   
        MRS                 0.00       0.00         0.00         0.00   
        PVC                 1.00       0.95         1.00         0.75   
        RA                  0.33       0.00         0.00         0.00   
        SARS                0.50       0.50         0.50         0.50   
        Staph               0.50       0.50         0.50         0.50   
        TNT                 0.50       0.50         0.50         0.50   
        US                  0.50       0.30         0.50         0.50   
term    B-Cell Leukemia     0.00       0.00         0.00         0.00   
        Borrelia            0.50       0.50         0.50         0.50   
        Callus              1.00       0.45         0.80         0.65   
        Cell                1.00       0.35         0.35         0.00   
        Cement              1.00       0.15         0.10         0.10   
        Cilia               1.00       0.45         0.50         0.65   
        ERUPTION            1.00       0.05         0.25         0.40   
        Exercises           1.00       0.25         0.25         0.25   
        Follicles           0.50       0.50         0.50         0.50   
        Ganglion            0.50       0.50         0.50         0.50   
        Medullary           1.00       0.30         0.00         0.00   
        Milk                1.00       0.05         0.05         0.50   
        Nurse               1.00       0.45         0.50         0.40   
        Phosphorus          0.50       0.50         0.15         0.15   
        Pneumocystis        0.00       0.00         0.00         0.00   
        Respiration         1.00       0.45         0.60         0.55   
        Sodium              0.50       0.50         0.50         0.50   
        Synapsis            0.50       0.50         0.50         0.50   
        Ventricles          1.00       0.65         0.65         0.85   

                         Wu-Palmer   Lin  Resnik (fixed)  Max lemma count  
group   word                                                               
acronym ADP                   0.70  0.60            0.60             0.50  
        ALS                   0.50  0.50            0.50             0.50  
        ANA                   0.00  0.00            0.00             0.00  
        CAD                   0.00  0.00            0.00             0.00  
        CAM                   0.00  0.00            0.00             0.00  
        CDA                   0.00  0.00            0.00             0.00  
        CI                    0.00  0.00            0.00             0.00  
        CLS                   0.00  0.00            0.00             0.00  
        Ca                    0.38  0.45            0.45             0.25  
        DDD                   0.00  0.00            0.00             0.00  
        DON                   0.00  0.00            0.00             0.00  
        FAS                   0.50  0.50   

**Observations:**
- Many words have **covered = 0** (e.g. CAD, DDD, GAG): all methods get 0, so these words only lower the average.
- Many words have **covered = 0.5** and **all methods at 0.5** (e.g. SARS, Sodium, Staph): only one of the two senses is in WordNet, so every method always chooses it. These words do not tell us which method is better.
- The interesting words are the ones with **covered = 1** (e.g. Callus, Cell, Milk, Ventricles). Here the methods really differ, and no method is the best on all words: Simple Lesk is the best on Callus (0.80), Cosine Lesk on Ventricles (0.85), Wu-Palmer on Milk (0.60).